# Etapa 3 · Punto 3 — Generación de candidatos con algoritmos de grafos

**Objetivo.** Usar la estructura del grafo de conocimiento (Neo4j) para identificar **canciones, artistas y álbumes
potencialmente relevantes para una preferencia dada**. Esa preferencia puede ser una canción, un artista, un tag o género,
un usuario de CritiqueBrainz o un texto libre.

Se combinan dos algoritmos complementarios:

| | Algoritmo | Parte del grafo que usa | Qué produce | Cómo contribuye a los candidatos |
|---|---|---|---|---|
| 1 | **Louvain** (detección de comunidades) | Co-grafo de **artistas** derivado de canciones similares (`SIMILAR_TO`), tags compartidos, críticos que reseñan a ambos y géneros/instrumentos/personas del bonus | Una **comunidad («escena») por artista** y la modularidad de la partición | Identifica la **escena** de la preferencia; en el modo «misma escena» **selecciona la comunidad** dentro de la cual PPR ordena; **anota y explica** cada candidato; queda en Neo4j (`Artist.community`) para consultas Cypher |
| 2 | **Personalized PageRank** (PPR) | Grafo heterogéneo completo: canciones, artistas, álbumes, tags, usuarios, países y entidades del bonus, con aristas ponderadas | Una **puntuación de cercanía** de cada nodo a las semillas: probabilidad estacionaria de una caminata aleatoria que reinicia en ellas | **Ordena** los candidatos de todos los tipos (canciones, artistas, álbumes) y respalda cada uno con un **camino explicativo** en el grafo |

Además, un análisis de **componentes conexas** (WCC) justifica cómo se construye la proyección.

**Flujo del notebook**

1. Proyección del grafo desde Neo4j: las aristas que se usan, sus pesos y la conectividad.
2. Algoritmo 1: Louvain sobre el co-grafo de artistas. Las comunidades se escriben en Neo4j (`Artist.community`).
3. Algoritmo 2: Personalized PageRank.
4. Combinación de ambos: la función `recomendar(preferencia)`.
5. Ejemplos con distintos tipos de preferencia.
6. Evaluación offline: se ocultan relaciones y se mide si el método las recupera, frente a líneas base y variantes.
7. Persistencia de recomendaciones por usuario: `(:User)-[:RECOMMENDED]->(:Track)`.

> **¿Por qué NetworkX y no Neo4j GDS?** AuraDB Free no incluye la librería Graph Data Science; en Aura solo se ofrece
> como servicio de pago, *Aura Graph Analytics*. Por eso se sigue el mecanismo del taller: **se proyecta el grafo desde
> Neo4j con Cypher**, los algoritmos se ejecutan en Python con NetworkX y **los resultados se vuelven a escribir en Neo4j**,
> igual que el modo `write` de GDS.
>
> Requiere haber ejecutado los notebooks `01` y `02`, y tener `entrega_3/.env`.

## 0. Entorno

In [1]:
%pip -q install "neo4j>=5.26,<7" "networkx>=3.2" "pandas>=2" "sentence-transformers>=3,<6"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, time, math, random, itertools
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import networkx as nx
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

DATA_DIR = Path(os.environ.get('DATA_DIR', '../data'))

# Parámetros del recomendador (justificados en la sección 6)
PESOS = {                 # peso de cada tipo de arista en la proyección (grafo no dirigido)
    'SUNG_BY': 1.0,       # canción - artista
    'SIMILAR_TO': 10.0,   # canción - canción; se multiplica por el score de similitud (0-1)
    'HAS_TAG': 1.0,       # canción - tag
    'APPEARS_ON': 1.0,    # canción - álbum      (punto 2)
    'RELEASED_BY': 1.0,   # álbum - artista      (punto 2)
    'REVIEWED': 1.0,      # usuario - entidad reseñada (punto 2)
    'ORIGIN': 0.2,        # artista - país: conecta casi todo, pero es una señal débil de gusto
    'BONUS': 1.0,         # artista - género / instrumento / persona (bonus de la Etapa 2)
}
ALPHA = 0.85              # probabilidad de continuar la caminata en PPR (1 - ALPHA = probabilidad de reinicio)
GAMMA = 0.3               # fracción de masa de reinicio que se asigna a la escena (Louvain) al ampliar semillas
N_CENTRALES = 20          # artistas centrales de la comunidad que se agregan como semillas
MAX_POR_ARTISTA = 2       # diversidad: máximo de canciones del mismo artista en la lista final
SEMILLA = 42

In [3]:
# Conexión a Neo4j (igual que en los puntos 1 y 2: credenciales desde entrega_3/.env)
from neo4j import GraphDatabase
from neo4j.exceptions import DriverError, ServiceUnavailable, SessionExpired
import logging
logging.getLogger('neo4j').setLevel(logging.CRITICAL)  # los cortes se reportan con nuestros propios mensajes de reintento

def cargar_env(ruta):
    for linea in ruta.read_text(encoding='utf-8').splitlines():
        if '=' in linea and not linea.lstrip().startswith('#'):
            k, v = linea.split('=', 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")

ARCHIVO_ENV = next((p for p in [Path('../.env'), Path('.env'), DATA_DIR / '.env'] if p.exists()), None)
if ARCHIVO_ENV is None:
    raise FileNotFoundError('No se encontró entrega_3/.env. Cópialo desde .env.example y completa las credenciales.')
cargar_env(ARCHIVO_ENV)
NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD = (os.environ[k] for k in ('NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD'))
NEO4J_DATABASE = os.environ.get('NEO4J_DATABASE') or None

def conectar(intentos=15, espera=5):
    for i in range(1, intentos + 1):
        d = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD),
                                 connection_timeout=15, max_transaction_retry_time=120,
                                 notifications_min_severity='OFF')
        try:
            d.verify_connectivity()
            return d
        except (ServiceUnavailable, SessionExpired) as e:
            d.close()
            if i == intentos:
                raise
            print(f'Aura no disponible ({type(e).__name__}); reintento {i}/{intentos - 1} en {espera}s...')
            time.sleep(espera)

def sesion():
    global driver
    try:
        return driver.session(database=NEO4J_DATABASE)
    except DriverError:
        driver = conectar()
        return driver.session(database=NEO4J_DATABASE)

def con_reintentos(fn, intentos=15, espera=5):
    # Ejecuta fn(session); si Aura corta la conexión, reconecta y reintenta (las escrituras son idempotentes)
    global driver
    for i in range(1, intentos + 1):
        try:
            with sesion() as session:
                return fn(session)
        except (ServiceUnavailable, SessionExpired) as e:
            if i == intentos:
                raise
            print(f'Conexión perdida ({type(e).__name__}); reconectando {i}/{intentos - 1} en {espera}s...')
            time.sleep(espera)
            try:
                driver.close()
            except Exception:
                pass
            driver = conectar()

def run_cypher(query, **parameters):
    return con_reintentos(lambda s: [dict(r) for r in s.run(query, **parameters)])

def cargar_en_lotes(nombre, query, rows, batch_size=1000):
    t = time.time()
    for i in range(0, len(rows), batch_size):
        lote = rows[i:i + batch_size]
        con_reintentos(lambda s: s.execute_write(lambda tx: tx.run(query, rows=lote).consume()))
    print(f'✓ {nombre:<40} {len(rows):>8,}  ({time.time() - t:.1f}s)')

driver = conectar()
print('Conexión establecida con', driver.get_server_info().agent, '| usuario', NEO4J_USER)
n_usuarios = run_cypher('MATCH (u:User) RETURN count(u) AS n')[0]['n']
assert n_usuarios > 0, 'Faltan los datos del punto 2: ejecuta primero 02_integracion_resenas_critiquebrainz.ipynb'

try:
    print('GDS disponible:', run_cypher('RETURN gds.version() AS v')[0]['v'])
except Exception as e:
    print('GDS no disponible en esta instancia →', str(e).split('message:')[-1][:80].strip(' {}'),
          '→ los algoritmos se ejecutan con NetworkX')

Conexión establecida con Neo4j/5.27-aura | usuario 4710b36a


GDS no disponible en esta instancia → Aura Graph Analytics is versionless.} {gql_status: 50N00} {gql_status_descripti → los algoritmos se ejecutan con NetworkX


## 1. Proyección del grafo desde Neo4j

Igual que en el taller, se extraen con Cypher las relaciones relevantes para recomendar y se arma un grafo de NetworkX.

| Arista en la proyección | Origen en Neo4j | Peso | Por qué |
|---|---|---|---|
| canción – artista | `SUNG_BY` | 1 | Une todas las canciones de un mismo artista |
| canción – canción | `SIMILAR_TO {score}` | 10 × score | Es la señal más directa de afinidad entre canciones. El factor 10 se justifica en 6.2 |
| canción – tag | `HAS_TAG` | 1 | El género o la etiqueta describen la canción |
| canción – álbum – artista | `APPEARS_ON`, `RELEASED_BY` | 1 | Contexto de lanzamiento, tomado del punto 2 |
| usuario – canción / álbum / artista | `(:User)<-[:WRITTEN_BY]-(:Review)-[:REVIEWS_*]->()` | 1 | **Preferencia de usuario**: un crítico conecta las entidades que reseñó |
| artista – país | `ORIGIN` | 0,2 | Conecta casi todo el grafo, pero dice poco del gusto. Por eso pesa poco (ver WCC abajo) |
| artista – género / instrumento / persona | relaciones del bonus | 1 | Conocimiento extraído de textos en la Etapa 2 |

**Qué se excluye:**
- **El meta-grafo de la ontología:** no son datos.
- **Los `Chunk`:** pertenecen al componente RAG.
- **`INTERACTED_WITH`:** se derivó de las mismas reseñas mediante las cadenas OWL. Si se incluyera, contaría dos veces
  el camino usuario → álbum → canción.

El nodo `Review` se colapsa en una arista usuario–entidad. El grafo es **no dirigido**, porque para recomendar hay que
recorrer las relaciones en ambos sentidos, por ejemplo canción → artista → otras canciones.

In [4]:
CONSULTAS = {
    'SUNG_BY':     'MATCH (s:Track)-[:SUNG_BY]->(o:Artist) RETURN s.uri AS s, o.uri AS o',
    'SIMILAR_TO':  'MATCH (s:Track)-[r:SIMILAR_TO]->(o:Track) RETURN s.uri AS s, o.uri AS o, r.score AS score',
    'HAS_TAG':     'MATCH (s:Track)-[:HAS_TAG]->(o:Tag) RETURN s.uri AS s, o.uri AS o',
    'APPEARS_ON':  'MATCH (s:Track)-[:APPEARS_ON]->(o:Album) RETURN s.uri AS s, o.uri AS o',
    'RELEASED_BY': 'MATCH (s:Album)-[:RELEASED_BY]->(o:Artist) RETURN s.uri AS s, o.uri AS o',
    'REVIEWED':    '''MATCH (s:User)<-[:WRITTEN_BY]-(r:Review)-[:REVIEWS_RECORDING|REVIEWS_ALBUM|REVIEWS_ARTIST]->(o)
                      RETURN s.uri AS s, o.uri AS o, r.rating AS rating''',
    'ORIGIN':      'MATCH (s:Artist)-[:ORIGIN]->(o:Country) RETURN s.uri AS s, o.uri AS o',
    'BONUS':       '''MATCH (s:Artist)-[r:ASSOCIATED_WITH_GENRE|INFLUENCED_BY_GENRE|PLAYS_INSTRUMENT|PRODUCED_BY|COLLABORATED_WITH]->(o)
                      RETURN s.uri AS s, o.uri AS o, type(r) AS rel''',
}
TIPOS = ['Track', 'Artist', 'Album', 'User', 'Tag', 'Country', 'Genre', 'Instrument', 'Person']   # Artist antes que Person

t0 = time.time()
DATOS = {rel: run_cypher(q) for rel, q in CONSULTAS.items()}
NODOS = pd.DataFrame(run_cypher('''
    MATCH (n) WHERE any(l IN labels(n) WHERE l IN $tipos)
    RETURN n.uri AS uri, [l IN $tipos WHERE l IN labels(n)][0] AS tipo, n.id AS id,
           coalesce(n.title, n.name, n.areas[0], n.id) AS nombre''', tipos=TIPOS))
print(f'Proyección descargada en {time.time() - t0:.0f}s')

TIPO = dict(zip(NODOS.uri, NODOS.tipo))
NOMBRE = dict(zip(NODOS.uri, NODOS.nombre.fillna('(sin nombre)')))
TRACK_URI = dict(zip(NODOS[NODOS.tipo == 'Track'].id, NODOS[NODOS.tipo == 'Track'].uri))
TRACK_ID = {u: i for i, u in TRACK_URI.items()}
ARTISTA_DE = {f['s']: f['o'] for f in DATOS['SUNG_BY']}          # canción -> artista
display(pd.DataFrame({'filas': {rel: len(v) for rel, v in DATOS.items()}}).T)
display(NODOS.tipo.value_counts().rename('nodos').to_frame().T)

Proyección descargada en 58s


,SUNG_BY,SIMILAR_TO,HAS_TAG,APPEARS_ON,RELEASED_BY,REVIEWED,ORIGIN,BONUS
filas,100000,6380,475,2981,1859,2006,22982,46


tipo,Track,Artist,Album,User,Tag,Country,Genre,Instrument,Person
nodos,100000,27378,1849,387,245,138,23,6,5


In [5]:
def construir_grafo(datos, pesos=PESOS, excluir=frozenset()):
    # Grafo heterogéneo no dirigido y ponderado; `excluir` = pares (s, o) que se omiten (para la evaluación)
    G = nx.Graph()
    G.add_nodes_from(TIPO)
    for rel, filas in datos.items():
        if not pesos.get(rel):
            continue
        for f in filas:
            s, o = f['s'], f['o']
            if (s, o) in excluir:
                continue
            w = pesos[rel] * (float(f['score'] or 0) if rel == 'SIMILAR_TO' else 1.0)
            if G.has_edge(s, o):            # p. ej. A SIMILAR_TO B y B SIMILAR_TO A
                G[s][o]['weight'] += w
            else:
                G.add_edge(s, o, weight=w, rel=f.get('rel', rel))
    return G

G = construir_grafo(DATOS)
print(f'Grafo proyectado: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas')

Grafo proyectado: 130,031 nodos, 136,721 aristas


### 1.1 Conectividad (componentes conexas, WCC)

Antes de elegir los algoritmos conviene ver **qué tan conectado** está el grafo. Si dos canciones están en componentes
distintas, ningún recorrido puede relacionarlas.

In [6]:
def resumen_componentes(H, nombre):
    comps = sorted(nx.connected_components(H), key=len, reverse=True)
    mayor = comps[0]
    tracks = [n for n in H if TIPO.get(n) == 'Track']
    return {'proyección': nombre, 'componentes': len(comps), 'nodos en la mayor': len(mayor),
            '% nodos en la mayor': round(100 * len(mayor) / H.number_of_nodes(), 1),
            '% canciones en la mayor': round(100 * sum(t in mayor for t in tracks) / len(tracks), 1)}

G_sin_pais = construir_grafo(DATOS, {**PESOS, 'ORIGIN': 0})
G_solo_items = construir_grafo(DATOS, {**PESOS, 'ORIGIN': 0, 'REVIEWED': 0})
display(pd.DataFrame([resumen_componentes(G, 'completa'),
                      resumen_componentes(G_sin_pais, 'sin país (ORIGIN)'),
                      resumen_componentes(G_solo_items, 'sin país ni usuarios')]).set_index('proyección'))
del G_sin_pais, G_solo_items

,componentes,nodos en la mayor,% nodos en la mayor,% canciones en la mayor
proyección,,,,
completa,4193,114906,88.4,89.3
sin país (ORIGIN),22831,33769,26.0,27.0
sin país ni usuarios,24112,26319,20.2,21.9


**Lectura:**
- **Sin el país**, la mayoría de las canciones queda aislada en un grupo pequeño con su propio artista: el dataset trae
  pocas relaciones `SIMILAR_TO` (6.380) y pocos tags (475).
- **El núcleo conectado** se forma con similitudes, tags y, sobre todo, **las reseñas de usuarios del punto 2**, que unen
  artistas reseñados por el mismo crítico.
- **El país** une casi todo el grafo, pero es una señal débil de gusto. Por eso entra con peso bajo (0,2): sirve de
  **respaldo** para canciones aisladas sin dominar el recorrido cuando existen señales más fuertes.

Esta dispersión también explica por qué se usan dos algoritmos. Louvain trabaja sobre el núcleo informativo, a nivel de
artista, para encontrar escenas. PPR recorre el grafo completo y aprovecha cualquier camino disponible.

## 2. Algoritmo 1 — Louvain: escenas musicales (detección de comunidades)

**Parte del grafo que usa.** Un **co-grafo de artistas** derivado del grafo de conocimiento. Dos artistas quedan unidos
cuando:
- tienen canciones relacionadas por `SIMILAR_TO`, con peso igual a la suma de los scores;
- comparten un **tag** en sus canciones;
- los reseñó el **mismo usuario** de CritiqueBrainz, es decir, la preferencia de usuario;
- comparten género, instrumento o persona del bonus.

En los nodos compartidos, cada par recibe peso `1/(n−1)`, donde `n` es el número de artistas conectados al nodo, al
estilo del índice *resource allocation*. Así, un tag muy genérico o un crítico que reseñó a 150 artistas no aplastan
las relaciones más específicas.

No se usa el país: crearía grupos enormes por nacionalidad, y la nacionalidad no es gusto musical. Tampoco se trabaja a
nivel de canción, porque ese grafo es demasiado disperso.

**Qué produce.** Una partición de los artistas en comunidades que maximiza la **modularidad**: hay más peso dentro de
cada comunidad del que se esperaría al azar. Cada comunidad funciona como una **escena** (pop/R&B, rock clásico, country,
blues, metal, hip-hop…). Se etiqueta con los tags más frecuentes de sus artistas.

**Cómo contribuye a los candidatos.**
- **Escena de la preferencia:** se toman las comunidades de los artistas de las semillas, ponderadas por su peso.
- **Modo «misma escena»:** Louvain selecciona la comunidad y PPR ordena las canciones dentro de ella. Es la
  combinación «comunidad + ordenamiento» que sugiere el enunciado, y sirve cuando se busca coherencia de estilo.
- **Explicación:** cada candidato se anota con su escena y se indica si es la misma de la preferencia.
- **Generador independiente:** como solo Louvain, ya es una señal fuerte (ver 6.1). Además queda guardado en Neo4j, así
  que los siguientes puntos pueden pedir candidatos de una escena solo con Cypher.
- **Ampliación de semillas (opcional, `ampliar=True`):** agrega los artistas centrales de la escena al vector de
  reinicio de PPR. Se evaluó en 6.1 y 6.3 y no mejoró el recall, así que está **desactivada por defecto**.

In [7]:
def construir_cografo(datos, excluir=frozenset()):
    A = nx.Graph()
    def sumar(a, b, w):
        if a and b and a != b:
            if A.has_edge(a, b):
                A[a][b]['weight'] += w
            else:
                A.add_edge(a, b, weight=w)
    album_artistas = defaultdict(set)
    for f in datos['RELEASED_BY']:
        album_artistas[f['s']].add(f['o'])
    def artistas(n):
        t = TIPO.get(n)
        return {n} if t == 'Artist' else album_artistas[n] if t == 'Album' else {ARTISTA_DE[n]} if t == 'Track' else set()
    for f in datos['SIMILAR_TO']:
        if (f['s'], f['o']) not in excluir:
            sumar(ARTISTA_DE[f['s']], ARTISTA_DE[f['o']], float(f['score'] or 0))
    grupos = defaultdict(set)                       # nodo compartido -> artistas conectados a él
    for f in datos['HAS_TAG']:
        if (f['s'], f['o']) not in excluir:
            grupos[f['o']].add(ARTISTA_DE[f['s']])
    for f in datos['REVIEWED']:
        if (f['s'], f['o']) not in excluir:
            grupos[f['s']] |= artistas(f['o'])
    for f in datos['BONUS']:
        if TIPO.get(f['o']) == 'Artist':
            sumar(f['s'], f['o'], 1.0)               # COLLABORATED_WITH: arista directa
        else:
            grupos[f['o']].add(f['s'])
    for miembros in grupos.values():
        if len(miembros) > 1:
            w = 1 / (len(miembros) - 1)
            for a, b in itertools.combinations(sorted(miembros), 2):
                sumar(a, b, w)
    return A

def detectar_comunidades(A, semilla=SEMILLA):
    comunidades = sorted(nx.community.louvain_communities(A, weight='weight', resolution=1.0, seed=semilla),
                         key=len, reverse=True)
    return {a: i for i, c in enumerate(comunidades) for a in c}, comunidades

t0 = time.time()
A = construir_cografo(DATOS)
COMUNIDAD, COMUNIDADES = detectar_comunidades(A)
FUERZA = dict(A.degree(weight='weight'))           # centralidad (grado ponderado) dentro del co-grafo
Q = nx.community.modularity(A, COMUNIDADES, weight='weight')
print(f'Co-grafo de artistas: {A.number_of_nodes():,} artistas, {A.number_of_edges():,} aristas')
print(f'Louvain: {len(COMUNIDADES)} comunidades, modularidad = {Q:.3f}  ({time.time() - t0:.1f}s)')
n_art = sum(t == 'Artist' for t in TIPO.values())
cubiertas = sum(1 for t in ARTISTA_DE.values() if t in COMUNIDAD)
print(f'Artistas con comunidad: {len(COMUNIDAD):,} de {n_art:,} | canciones cuyo artista tiene comunidad: '
      f'{cubiertas:,} de {len(ARTISTA_DE):,} ({cubiertas / len(ARTISTA_DE):.1%})')

Co-grafo de artistas: 4,747 artistas, 34,237 aristas
Louvain: 133 comunidades, modularidad = 0.812  (0.2s)
Artistas con comunidad: 4,747 de 27,378 | canciones cuyo artista tiene comunidad: 28,130 de 100,000 (28.1%)


In [8]:
TAGS_ARTISTA = defaultdict(Counter)
for f in DATOS['HAS_TAG']:
    TAGS_ARTISTA[ARTISTA_DE[f['s']]][NOMBRE[f['o']].lower()] += 1
for f in DATOS['BONUS']:
    if TIPO.get(f['o']) == 'Genre':
        TAGS_ARTISTA[f['s']][NOMBRE[f['o']].lower()] += 1

def describir_comunidades(comunidades, fuerza):
    filas = []
    for c, miembros in enumerate(comunidades):
        tags = Counter()                            # nº de artistas de la comunidad que usan cada tag
        for a in miembros:
            tags.update({t for t in TAGS_ARTISTA[a] if len(t) <= 20})
        tags = Counter({t: n for t, n in tags.items() if n >= 2})
        centrales = sorted(miembros, key=lambda a: -fuerza.get(a, 0))[:6]
        filas.append({'comunidad': c, 'artistas': len(miembros),
                      'etiqueta': ' · '.join(t for t, _ in tags.most_common(3)) or '(sin tags)',
                      'artistas centrales': ', '.join(NOMBRE[a] for a in centrales)})
    return pd.DataFrame(filas).set_index('comunidad')

DESC_COM = describir_comunidades(COMUNIDADES, FUERZA)
ETIQUETA_COM = DESC_COM.etiqueta.to_dict()
display(DESC_COM.head(15))
print('Tamaño de las comunidades:', DESC_COM.artistas.describe()[['min', '50%', 'max']].to_dict())

,artistas,etiqueta,artistas centrales
comunidad,,,
0,380,rock · hard rock · folk,"Kings Of Leon, Aerosmith, Los Fabulosos, Top Billin, Bruce Springsteen, Paul Rogers"
1,350,pop · female vocalists · latin,"Gwen Stefani, Lyala, Rihanna, Akon, Arctic Monkeys, Sid Le Rock"
2,346,heavy metal · nu metal · hip-hop,"Mary, Radiohead, Salt-N-Pepa, Massive Attack, Red Hot Chili Peppers, U2"
3,231,jazz · hip-hop · hip hop,"Birdapres, Dilated Peoples, OutKast, Busta Rhymes, King Curtis, A Tribe Called Quest"
4,215,electronic · indie · easy listening,"Franz Ferdinand, Faithless, Prodigy, Talvin Singh, Usher, Belle & Sebastian"
5,195,country,"The Jordanaires, Marty Robbins, Eddy Arnold, K.T. Oslin, Eddy Raven, Hank Locklin"
6,168,blues,"Eddie Jones, Mississippi Fred McDowell, Washboard Sam, Junior Kimbrough, Blind Willie Johnson, Otis Spann"
7,168,soul · pop,"Little Eva, Fontella Bass, The Four Pennies, The Ronettes, Minnie Riperton, Jimmy Hughes"
8,163,industrial,"Nine Inch Nails, Martina McBride, Patty Griffin, Fuck Buttons, Miranda Lambert, Soulive"


Tamaño de las comunidades: {'min': 2.0, '50%': 6.0, 'max': 380.0}


Las comunidades son coherentes musicalmente: aparecen escenas de pop/R&B, rock clásico, country, blues de Delta, metal y
hip-hop. Hay que recordar que la partición no usa los tags como etiqueta de entrada. Los tags son solo una de las
fuentes de aristas, y la mayoría de los artistas no tiene ninguno.

**Escritura en Neo4j** (equivalente al modo `write` de GDS): `(:Artist {community})`. Así la escena queda disponible
desde Cypher para los puntos siguientes.

In [9]:
run_cypher('MATCH (a:Artist) WHERE a.community IS NOT NULL REMOVE a.community')
cargar_en_lotes('Artist.community (Louvain)', 'UNWIND $rows AS row MATCH (a:Artist {uri: row.uri}) SET a.community = row.c',
                [{'uri': a, 'c': c} for a, c in COMUNIDAD.items()])
display(pd.DataFrame(run_cypher('''
    MATCH (a:Artist) WHERE a.community IS NOT NULL
    WITH a.community AS comunidad, count(*) AS artistas, collect(a.name)[..5] AS ejemplo
    RETURN comunidad, artistas, ejemplo ORDER BY artistas DESC LIMIT 5''')))

Conexión perdida (ServiceUnavailable); reconectando 1/14 en 5s...


Aura no disponible (ServiceUnavailable); reintento 1/14 en 5s...


Aura no disponible (ServiceUnavailable); reintento 2/14 en 5s...


Aura no disponible (ServiceUnavailable); reintento 3/14 en 5s...


Aura no disponible (SessionExpired); reintento 4/14 en 5s...


Aura no disponible (ServiceUnavailable); reintento 5/14 en 5s...


Aura no disponible (ServiceUnavailable); reintento 6/14 en 5s...


✓ Artist.community (Louvain)                  4,747  (126.3s)


,comunidad,artistas,ejemplo
0,0,380,"[Upsilon Acrux, Crippled Black Phoenix, Maroon 5, Atomic Swing, Lisa Loeb & Nine Stories]"
1,1,350,"[Herborg Kråkevik, Marcela Morelo, CeCe Peniston, Five for Fighting, Lovespirals]"
2,2,346,"[Spinnerette, Evan Dando, Dizzee Rascal Feat. Calvin Harris & Chrome, Smash hit combo, MC Serch]"
3,3,231,"[Creutzfeld & Jakob, Marco Polo & Torae, Bill Bruford's Earthworks, Lordz Of Brooklyn, Crash Crew]"
4,4,215,"[Jim Hart's Gemini, Kinobe, Galactic Caravan, Dog Day, Postal Service]"


## 3. Algoritmo 2 — Personalized PageRank (PPR)

**Parte del grafo que usa.** El grafo heterogéneo completo de la sección 1, con sus pesos.

**Qué produce.** Para un conjunto de **semillas** (los nodos de la preferencia, cada uno con un peso), la distribución
estacionaria de una caminata aleatoria que en cada paso:
- con probabilidad `α = 0,85` sigue una arista, elegida en proporción a su peso;
- con probabilidad `1 − α` vuelve a una semilla.

El resultado es una **puntuación de cercanía estructural** para *cada* nodo. Un nodo puntúa alto cuando hay muchos
caminos cortos y fuertes desde las semillas hasta él. Los nodos muy conectados no ganan solo por ser populares, porque
el reinicio mantiene la caminata cerca de las semillas.

**Cómo contribuye a los candidatos.**
- **Ordena por relevancia** canciones, artistas y álbumes en una sola ejecución.
- **Admite cualquier tipo de preferencia** y varias semillas a la vez, por ejemplo un usuario más un tag.
- **Combina varias señales** al mismo tiempo: similitud, artista, álbum, tags, críticos en común y país.
- **Respalda cada recomendación con el camino** más fuerte desde las semillas, que sirve de explicación.

In [10]:
def ppr(ctx, semillas, alpha=ALPHA):
    pers = {n: w for n, w in semillas.items() if n in ctx['G'] and w > 0}
    if not pers:
        raise ValueError('Ninguna semilla está en el grafo')
    return nx.pagerank(ctx['G'], alpha=alpha, personalization=pers, weight='weight')

def etiqueta(n):
    t = TIPO.get(n)
    if t == 'Track':
        return f'«{NOMBRE[n]}» ({NOMBRE[ARTISTA_DE[n]]})'
    return {'Artist': '{}', 'Album': 'álbum «{}»', 'Tag': '#{}', 'Genre': 'género {}', 'User': 'usuario {}',
            'Country': 'país {}'}.get(t, '{} (' + str(t) + ')').format(NOMBRE.get(n))

CTX = {'G': G, 'A': A, 'com': COMUNIDAD, 'comms': COMUNIDADES, 'fuerza': FUERZA}

# Ejemplo: la canción del dataset con más relaciones de similitud
salientes = Counter(f['s'] for f in DATOS['SIMILAR_TO'])
ejemplo = salientes.most_common(1)[0][0]
t0 = time.time()
sc = ppr(CTX, {ejemplo: 1.0})
print(f'Semilla: {etiqueta(ejemplo)} — PPR sobre {G.number_of_nodes():,} nodos en {time.time() - t0:.2f}s')
top = sorted(((n, s) for n, s in sc.items() if n != ejemplo), key=lambda x: -x[1])
for tipo in ('Track', 'Artist', 'Tag', 'Country'):
    print(f'\nTop {tipo}:')
    for n, s in [x for x in top if TIPO.get(x[0]) == tipo][:5]:
        print(f'  {s:.5f}  {etiqueta(n)}')

Semilla: «The Loco-Motion» (Little Eva) — PPR sobre 130,031 nodos en 0.24s

Top Track:
  0.04122  «Rescue Me» (Fontella Bass)
  0.00881  «Sweeter Than You» (The Four Pennies)
  0.00863  «It's My Party» (Lesley Gore)
  0.00783  «Under The Boardwalk» (Rhythms Del Mundo feat. Eros Ramazzotti / Eros Ramazzotti)
  0.00762  «Runaway» (Del Shannon)

Top Artist:
  0.00379  Danny & The Juniors
  0.00310  Jackie Wilson
  0.00266  Chordettes
  0.00265  Del Vikings
  0.00261  Little Anthony & The Imperials

Top Tag:
  0.00112  #oldies
  0.00025  #soul
  0.00024  #pop
  0.00019  #60s
  0.00002  #1959

Top Country:
  0.00402  país Alabama
  0.00069  país Abingdon-on-Thames
  0.00020  país Baden-Württemberg
  0.00014  país Brescia
  0.00012  país Bayeux


## 4. Combinación: `recomendar(preferencia)`

La preferencia se expresa con cualquiera de estos campos, combinables entre sí:

- `canciones`: IDs MSD (`TR…`) o títulos;
- `artistas`: nombres;
- `tags`: tags o géneros;
- `usuario`: un usuario de CritiqueBrainz;
- `texto`: una descripción libre. Se resuelve con el **índice vectorial del punto 2**: los fragmentos de reseñas más
  parecidos al texto apuntan a las entidades reseñadas, que pasan a ser semillas ponderadas por similitud.

Pasos de la recomendación:

1. **Semillas:** los nodos de la preferencia, con peso.
2. **Escena (Louvain):** las comunidades de los artistas de las semillas.
3. **Ampliación (opcional, Louvain → PPR):** con `ampliar=True`, o con `'auto'` solo para preferencias débiles,
   se agregan los `N_CENTRALES` artistas más centrales de la escena principal con una fracción `GAMMA` de la masa de
   reinicio. Por defecto está desactivada (ver la evaluación).
4. **PPR:** puntúa todos los nodos.
5. **Selección de candidatos:**
   - las canciones mejor puntuadas, sin las ya conocidas (las semillas y, si la preferencia es un usuario, lo que ya
     reseñó);
   - diversidad: máximo `MAX_POR_ARTISTA` canciones por artista, y sin duplicados del MSD (la misma canción con
     otro ID);
   - opcionalmente, `modo='escena'` para quedarse solo con canciones de la escena.
   Se seleccionan además artistas y álbumes.
6. **Explicación:** el camino más fuerte en el grafo desde una semilla hasta cada candidato (Dijkstra con costo
   `1/peso`) y la escena del candidato.

In [11]:
def artistas_de(ctx, n):
    G, t = ctx['G'], TIPO.get(n)
    if t == 'Artist':
        return [n]
    if t in ('Track', 'Album'):
        return [v for v in G[n] if TIPO.get(v) == 'Artist']
    if t == 'User':
        return [a for v in G[n] for a in artistas_de(ctx, v)]
    if t in ('Tag', 'Genre', 'Instrument', 'Person'):
        return [a for v in G[n] for a in ([v] if TIPO.get(v) == 'Artist' else artistas_de(ctx, v) if TIPO.get(v) == 'Track' else [])]
    return []

def escena(ctx, semillas):
    # Comunidades (Louvain) de los artistas de las semillas, ponderadas por el peso de cada semilla
    c = Counter()
    for n, w in semillas.items():
        arts = [a for a in artistas_de(ctx, n) if a in ctx['com']]
        for a in arts:
            c[ctx['com'][a]] += w / len(arts)
    return c

def ampliar_semillas(ctx, semillas, esc, gamma=GAMMA, n=N_CENTRALES):
    if not esc:
        return dict(semillas)
    principal = esc.most_common(1)[0][0]
    centrales = sorted(ctx['comms'][principal], key=lambda a: -ctx['fuerza'].get(a, 0))[:n]
    total = sum(semillas.values())
    nuevas = {k: v * (1 - gamma) for k, v in semillas.items()}
    for a in centrales:
        nuevas[a] = nuevas.get(a, 0) + gamma * total / len(centrales)
    return nuevas

def grado_util(ctx, n):
    # Vecinos que aportan señal de gusto (no cuenta el país)
    return sum(1 for v in ctx['G'][n] if TIPO.get(v) != 'Country')

def canciones_conocidas(ctx, semillas):
    G, conocidas = ctx['G'], set()
    for n in semillas:
        t = TIPO.get(n)
        if t == 'Track':
            conocidas.add(n)
        elif t == 'User':                            # lo que el usuario ya reseñó
            for v in G[n]:
                conocidas |= {v} if TIPO.get(v) == 'Track' else {x for x in G[v] if TIPO.get(x) == 'Track'}
    return conocidas

def titulo_base(titulo):
    return re.sub(r'\s*[\(\[].*?[\)\]]', '', str(titulo)).strip().casefold()

def rankear_canciones(ctx, scores, conocidas=frozenset(), esc=None, modo='global', max_por_artista=None):
    cands = sorted(((n, s) for n, s in scores.items() if s > 0 and TIPO.get(n) == 'Track' and n not in conocidas),
                   key=lambda x: -x[1])
    if modo == 'escena' and esc:
        tope = max(esc.values())
        comunidades = {c for c, w in esc.items() if w >= 0.25 * tope}
        cands = [(n, s) for n, s in cands if ctx['com'].get(ARTISTA_DE[n]) in comunidades]
    if max_por_artista:                              # diversidad + sin duplicados del MSD (misma canción, otro ID)
        cuenta, vistas, salida = Counter(), set(), []
        for n, s in cands:
            a, clave = ARTISTA_DE[n], (ARTISTA_DE[n], titulo_base(NOMBRE[n]))
            if cuenta[a] < max_por_artista and clave not in vistas:
                salida.append((n, s))
                cuenta[a] += 1
                vistas.add(clave)
        cands = salida
    return cands

def explicar(ctx, semillas, destino):
    try:
        _, camino = nx.multi_source_dijkstra(ctx['G'], set(semillas), target=destino,
                                             weight=lambda u, v, d: 1.0 / d['weight'])
    except nx.NetworkXNoPath:
        return '(sin camino: llega por la escena)'
    partes = [etiqueta(camino[0])]
    for u, v in zip(camino, camino[1:]):
        partes.append(f'─{ctx["G"][u][v]["rel"]}─ {etiqueta(v)}')
    return ' '.join(partes)

In [12]:
_norm = lambda s: re.sub(r'\s+', ' ', str(s)).strip().casefold()
INDICE_NOMBRE = defaultdict(list)
for u, t in TIPO.items():
    INDICE_NOMBRE[(t, _norm(NOMBRE[u]))].append(u)

def buscar(nombre, tipos):
    cands = [u for t in tipos for u in INDICE_NOMBRE.get((t, _norm(nombre)), [])]
    if not cands:                                   # coincidencia parcial
        n = _norm(nombre)
        cands = [u for (t, nm), us in INDICE_NOMBRE.items() if t in tipos and n in nm for u in us]
    if not cands:
        raise ValueError(f'No se encontró «{nombre}» entre {"/".join(tipos)}')
    return max(cands, key=lambda u: G.degree(u, weight='weight'))   # si hay homónimos: el más conectado

_modelo = None
def semillas_desde_texto(texto, k=10, temperatura=0.02):
    # Texto libre -> índice vectorial del punto 2 -> entidades reseñadas (semillas ponderadas por similitud)
    global _modelo
    if _modelo is None:
        from sentence_transformers import SentenceTransformer
        _modelo = SentenceTransformer('intfloat/multilingual-e5-small')
    qv = _modelo.encode(f'query: {texto}', normalize_embeddings=True).tolist()
    filas = run_cypher('''
        CALL db.index.vector.queryNodes('chunk_embedding', $k, $qv) YIELD node AS c, score
        MATCH (c)-[:PART_OF]->(:Review)-[:REVIEWS_RECORDING|REVIEWS_ALBUM|REVIEWS_ARTIST]->(e)
        RETURN e.uri AS uri, max(score) AS score ORDER BY score DESC''', k=k, qv=qv)
    tope = filas[0]['score']
    return {f['uri']: math.exp((f['score'] - tope) / temperatura) for f in filas if f['uri'] in TIPO}

def resolver_preferencia(canciones=(), artistas=(), tags=(), usuario=None, texto=None):
    semillas = Counter()
    for c in canciones:
        semillas[TRACK_URI[c] if re.fullmatch(r'TR[A-Z0-9]{16}', c) else buscar(c, ['Track'])] += 1
    for a in artistas:
        semillas[buscar(a, ['Artist'])] += 1
    for t in tags:
        semillas[buscar(t, ['Tag', 'Genre'])] += 1
    if usuario:
        semillas[usuario if usuario in TIPO else buscar(usuario, ['User'])] += 1
    if texto:
        for u, w in semillas_desde_texto(texto).items():
            semillas[u] += w
    if not semillas:
        raise ValueError('La preferencia está vacía')
    return dict(semillas)

In [13]:
def recomendar(canciones=(), artistas=(), tags=(), usuario=None, texto=None, k=10, modo='global',
               ampliar=False, max_por_artista=MAX_POR_ARTISTA, explicaciones=True, ctx=None, mostrar=True):
    ctx = ctx or CTX
    semillas = resolver_preferencia(canciones, artistas, tags, usuario, texto)
    esc = escena(ctx, semillas)
    debil = bool(tags or texto) or sum(grado_util(ctx, n) for n in semillas) < 5
    usar_ampliacion = debil if ampliar == 'auto' else bool(ampliar)   # 'auto': solo preferencias débiles
    pers = ampliar_semillas(ctx, semillas, esc) if usar_ampliacion else semillas
    scores = ppr(ctx, pers)

    elegidas = rankear_canciones(ctx, scores, canciones_conocidas(ctx, semillas), esc, modo, max_por_artista)[:k]
    tope = elegidas[0][1] if elegidas else 1
    principal = esc.most_common(1)[0][0] if esc else None
    canciones_df = pd.DataFrame([{
        'canción': NOMBRE[n], 'artista': NOMBRE[ARTISTA_DE[n]], 'score': round(s / tope, 3),
        'escena': ETIQUETA_COM.get(ctx['com'].get(ARTISTA_DE[n]), '—') if ctx is CTX else ctx['com'].get(ARTISTA_DE[n]),
        'misma escena': ctx['com'].get(ARTISTA_DE[n]) == principal and principal is not None,
        **({'por qué (camino más fuerte)': explicar(ctx, semillas, n)} if explicaciones else {}),
        'uri': n} for n, s in elegidas], index=pd.RangeIndex(1, len(elegidas) + 1, name='rank'))

    excl_art = {a for n in semillas for a in artistas_de(ctx, n)}
    def top_tipo(tipo, excluir):
        filas = sorted(((n, s) for n, s in scores.items() if TIPO.get(n) == tipo and n not in excluir and n not in semillas),
                       key=lambda x: -x[1])[:k]
        return pd.DataFrame([{tipo.lower(): NOMBRE[n], 'score': round(s / filas[0][1], 3),
                              **({'escena': ETIQUETA_COM.get(ctx['com'].get(n), '—')} if tipo == 'Artist' else {})}
                             for n, s in filas], index=pd.RangeIndex(1, len(filas) + 1, name='rank'))
    artistas_df = top_tipo('Artist', excl_art)
    albumes_df = top_tipo('Album', set())

    if mostrar:
        print('Semillas:', '; '.join(f'{etiqueta(n)} [{w:.2f}]' for n, w in sorted(semillas.items(), key=lambda x: -x[1])[:6]),
              '…' if len(semillas) > 6 else '')
        print('Escena (Louvain):', '; '.join(f'{c}: {ETIQUETA_COM.get(c, c)} ({w / sum(esc.values()):.0%})'
                                             for c, w in esc.most_common(3)) if esc else 'ninguna (artistas fuera del núcleo)')
        print('Ampliación de semillas con la escena:', 'sí' if usar_ampliacion and esc else 'no', '| modo:', modo)
    return {'canciones': canciones_df, 'artistas': artistas_df, 'albumes': albumes_df,
            'semillas': semillas, 'escena': esc, 'scores': scores}

## 5. Ejemplos con distintos tipos de preferencia

### 5.1 Una canción

In [14]:
cancion_demo = max((t for t in TRACK_URI.values() if NOMBRE[ARTISTA_DE[t]] == 'Metallica'), key=lambda t: G.degree(t))
r = recomendar(canciones=[TRACK_ID[cancion_demo]])
display(r['canciones'].drop(columns='uri'))

Semillas: «The End Of The Line» (Metallica) [1.00] 
Escena (Louvain): 14: metalcore · thrash metal · grunge (100%)
Ampliación de semillas con la escena: no | modo: global


,canción,artista,score,escena,misma escena,por qué (camino más fuerte)
rank,,,,,,
1,I Am The Law,Anthrax,1.000,black metal · metal · gothic metal,False,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «I Am The Law» (Anthrax)
2,Angel,Aerosmith,0.877,rock · hard rock · folk,False,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «Crazy Train» (Ozzy Osbourne) ─SIMILAR_TO─ «Angel» (Aerosmith)
3,Iron Man,Black Sabbath,0.634,rock · hard rock · folk,False,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «Iron Man» (Black Sabbath)
4,Crazy Train,Ozzy Osbourne,0.627,rock · hard rock · folk,False,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «Crazy Train» (Ozzy Osbourne)
5,Ace of spades,Motorhead,0.592,heavy metal · nu metal · hip-hop,False,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «Ace of spades» (Motorhead)
6,Rainbow In The Dark,Dio,0.535,rock · hard rock · folk,False,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «Rainbow In The Dark» (Dio)
7,Hard to See,Five Finger Death Punch,0.515,metalcore · thrash metal · grunge,True,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «Hard to See» (Five Finger Death Punch)
8,I Stand Alone,Godsmack,0.486,metalcore · thrash metal · grunge,True,«The End Of The Line» (Metallica) ─SIMILAR_TO─ «I Stand Alone» (Godsmack)
9,Within My Blood,Skeletonwitch,0.446,metalcore · thrash metal · grunge,True,«The End Of The Line» (Metallica) ─HAS_TAG─ #thrash metal ─HAS_TAG─ «Within My Blood» (Skeletonwitch)


### 5.2 Un artista (con artistas recomendados)

In [15]:
r = recomendar(artistas=['Bukka White'])
display(r['canciones'].drop(columns=['uri', 'por qué (camino más fuerte)']))
display(r['artistas'])

Semillas: Bukka White [1.00] 
Escena (Louvain): 6: blues (100%)
Ampliación de semillas con la escena: no | modo: global


,canción,artista,score,escena,misma escena
rank,,,,,
1,Single Man Blues,Bukka White,1.000,blues,True
2,Bald Eagle Train,Bukka White,0.456,blues,True
3,Bucket's Got A Hole In It,Washboard Sam,0.184,blues,True
4,You Gotta Move,Mississippi Fred McDowell,0.174,blues,True
5,34 Blues,Charly Patton,0.146,blues,True
6,Meet Me in the City,Junior Kimbrough,0.124,blues,True
7,Gone Dead On You Blues,Blind Lemon Jefferson,0.097,blues,True
8,The Things That I Used To Do,Eddie Jones,0.092,blues,True
9,Precious Lord,Blind Boy Fuller,0.082,blues,True


,artist,score,escena
rank,,,
1,Son House,1.000,blues
2,Mississippi John Hurt,0.748,blues
3,Leadbelly,0.694,blues
4,Robert Johnson,0.635,blues
5,Charly Patton,0.579,blues
6,Furry Lewis,0.529,blues
7,Otis Spann,0.453,blues
8,Jesse Fuller,0.453,blues
9,Robert Pete Williams,0.427,blues


### 5.3 Un tag, con y sin ampliación por la escena

El tag `blues` está en pocas canciones. Louvain identifica su escena (blues del Delta). Se compara el resultado de PPR
solo con el de PPR ampliado con los artistas centrales de esa escena.

In [16]:
r_sin = recomendar(tags=['blues'], explicaciones=False)
r_amp = recomendar(tags=['blues'], ampliar=True, explicaciones=False, mostrar=False)
display(pd.concat({'con escena (Louvain + PPR)': r_amp['canciones'][['canción', 'artista', 'escena']],
                   'solo PPR': r_sin['canciones'][['canción', 'artista', 'escena']]}, axis=1))

Semillas: #blues [1.00] 
Escena (Louvain): 6: blues (100%)
Ampliación de semillas con la escena: no | modo: global


con escena (Louvain + PPR)                                                        solo PPR                                  
                           canción                    artista escena                       canción                    artista escena
rank                                                                                                                                
1     The Things That I Used To Do                Eddie Jones  blues  The Things That I Used To Do                Eddie Jones  blues
2                   You Gotta Move  Mississippi Fred McDowell  blues                You Gotta Move  Mississippi Fred McDowell  blues
3        Bucket's Got A Hole In It              Washboard Sam  blues     Bucket's Got A Hole In It              Washboard Sam  blues
4           Gone Dead On You Blues      Blind Lemon Jefferson  blues        Gone Dead On You Blues      Blind Lemon Jefferson  blues
5              Meet Me in the City           Junior Kimbrough  blues           Meet Me in the City           Junior Kimbrough  blues
6                   No Consolation               Bob Margolin  blues            John the Revelator       Blind Willie Johnson  blues
7                    Precious Lord           Blind Boy Fuller  blues                 Precious Lord           Blind Boy Fuller  blues
8               John the Revelator       Blind Willie Johnson  blues                      34 Blues              Charly Patton  blues
9                         34 Blues              Charly Patton  blues                No Consolation               Bob Margolin  blues
10     Hillbillies From Outerspace       The Vaughan Brothers  blues                     Evil Ways                 Otis Spann  blues

### 5.4 Un usuario de CritiqueBrainz (preferencias del punto 2)

In [17]:
usuarios = Counter(f['s'] for f in DATOS['REVIEWED'])
usuario_demo = next(u for u, n in usuarios.most_common() if 5 <= n <= 30)
print('Usuario:', NOMBRE[usuario_demo], '| entidades reseñadas:', usuarios[usuario_demo])
print('Reseñó:', ', '.join(etiqueta(f['o']) for f in DATOS['REVIEWED'] if f['s'] == usuario_demo)[:400], '\n')
r = recomendar(usuario=usuario_demo)
display(r['canciones'].drop(columns='uri'))
display(r['albumes'].head(5))

Usuario: tgggd86 | entidades reseñadas: 27
Reseñó: «Bleed Black» (AFI), «The Leaving Song» (AFI), «Sweet Emotion» (Aerosmith), «Lie To Me (Acoustic)» (12 Stones), «Wasteland» (10 Years), «Citizen/Soldier» (3 Doors Down), «Weatherman» (+44), «When Your Heart Stops Beating» (+44), álbum «Sing the Sorrow», álbum «Toys in the Attic», álbum «Planets», álbum «Unstable», álbum «311», álbum «Back in Blood», álbum «Paris Kills», álbum «Wasting the Dawn», á 



Semillas: usuario tgggd86 [1.00] 
Escena (Louvain): 0: rock · hard rock · folk (100%)
Ampliación de semillas con la escena: no | modo: global


,canción,artista,score,escena,misma escena,por qué (camino más fuerte)
rank,,,,,,
1,Rock Steady,Bad Company,1.000,classic rock · christmas,False,usuario tgggd86 ─REVIEWED─ «Sweet Emotion» (Aerosmith) ─SIMILAR_TO─ «Rock Steady» (Bad Company)
2,Kryptonite,3 Doors Down,0.602,rock · hard rock · folk,True,usuario tgggd86 ─REVIEWED─ «Citizen/Soldier» (3 Doors Down) ─SUNG_BY─ 3 Doors Down ─SUNG_BY─ «Kryptonite» (3 Doors D...
3,Here Without You,3 Doors Down,0.596,rock · hard rock · folk,True,usuario tgggd86 ─REVIEWED─ «Citizen/Soldier» (3 Doors Down) ─SUNG_BY─ 3 Doors Down ─SUNG_BY─ «Here Without You» (3 D...
4,Time Is On My Side,The Rolling Stones,0.467,classic rock · christmas,False,usuario tgggd86 ─REVIEWED─ «Sweet Emotion» (Aerosmith) ─SIMILAR_TO─ «Time Is On My Side» (The Rolling Stones)
5,3 Leaf Loser,12 Stones,0.398,rock · hard rock · folk,True,usuario tgggd86 ─REVIEWED─ «Lie To Me (Acoustic)» (12 Stones) ─SUNG_BY─ 12 Stones ─SUNG_BY─ «3 Leaf Loser» (12 Stones)
6,In Closing,12 Stones,0.398,rock · hard rock · folk,True,usuario tgggd86 ─REVIEWED─ «Lie To Me (Acoustic)» (12 Stones) ─SUNG_BY─ 12 Stones ─SUNG_BY─ «In Closing» (12 Stones)
7,Can't Get Enough,Bad Company,0.382,classic rock · christmas,False,usuario tgggd86 ─REVIEWED─ «Sweet Emotion» (Aerosmith) ─SIMILAR_TO─ «Rock Steady» (Bad Company) ─SIMILAR_TO─ «Can't ...
8,Angel,Aerosmith,0.238,rock · hard rock · folk,True,usuario tgggd86 ─REVIEWED─ «Sweet Emotion» (Aerosmith) ─SUNG_BY─ Aerosmith ─SUNG_BY─ «Angel» (Aerosmith)
9,Way Beyond,32 Leaves,0.194,rock · hard rock · folk,True,usuario tgggd86 ─REVIEWED─ «Sweet Emotion» (Aerosmith) ─SUNG_BY─ Aerosmith ─SUNG_BY─ «Angel» (Aerosmith) ─HAS_TAG─ #...


,album,score
rank,,
1,When Your Heart Stops Beating,1.000
2,The Autumn Effect,0.874
3,12 Stones,0.731
4,Sing the Sorrow,0.685
5,Division,0.678


### 5.5 Texto libre: puente con el corpus RAG del punto 2

La consulta en español se resuelve contra los fragmentos de reseñas, que están en inglés, mediante el índice vectorial.
Las entidades reseñadas pasan a ser semillas de PPR.

In [18]:
r = recomendar(texto='guitarras acústicas melancólicas, folk íntimo con voz suave', explicaciones=False)
display(r['canciones'].drop(columns='uri'))

W1118 14:22:31.883000 8554 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Conexión perdida (ServiceUnavailable); reconectando 1/14 en 5s...


Aura no disponible (ServiceUnavailable); reintento 1/14 en 5s...


Aura no disponible (ServiceUnavailable); reintento 2/14 en 5s...


Semillas: álbum «Tourist» [1.00]; álbum «Natural» [0.99]; álbum «Holes in the Wall» [0.98]; álbum «Democustico» [0.97]; álbum «The Hard Way» [0.95]; álbum «Middle Cyclone» [0.95] …
Escena (Louvain): 21: (sin tags) (21%); 18: (sin tags) (20%); 2: heavy metal · nu metal · hip-hop (20%)
Ampliación de semillas con la escena: no | modo: global


,canción,artista,score,escena,misma escena
rank,,,,,
1,PÍra,Democustico,1.000,(sin tags),True
2,Hand It Over,James Hunter,0.695,(sin tags),False
3,Slow Motion Bossa Nova,Celso Fonseca,0.551,(sin tags),False
4,A Origem Da Felicidade,Celso Fonseca,0.551,(sin tags),False
5,Gliss,Asobi Seksu,0.500,rock · hard rock · folk,False
6,New Years,Asobi Seksu,0.500,rock · hard rock · folk,False
7,This Given Line,The Electric Soft Parade,0.494,electronic · indie · easy listening,False
8,Silent To The Dark,The Electric Soft Parade,0.494,electronic · indie · easy listening,False
9,Wires (Album Version),Athlete,0.459,(sin tags),True


### 5.6 Modo «misma escena»: Louvain filtra y PPR ordena dentro de la comunidad

In [19]:
r_g = recomendar(canciones=[TRACK_ID[cancion_demo]], explicaciones=False, mostrar=False)
r_e = recomendar(canciones=[TRACK_ID[cancion_demo]], modo='escena', explicaciones=False)
display(pd.concat({'global': r_g['canciones'][['canción', 'artista', 'escena']],
                   'misma escena': r_e['canciones'][['canción', 'artista', 'escena']]}, axis=1))

Semillas: «The End Of The Line» (Metallica) [1.00] 
Escena (Louvain): 14: metalcore · thrash metal · grunge (100%)
Ampliación de semillas con la escena: no | modo: escena


global                                                                              misma escena                                                            
                  canción                  artista                              escena                      canción                  artista                             escena
rank                                                                                                                                                                           
1            I Am The Law                  Anthrax  black metal · metal · gothic metal                  Hard to See  Five Finger Death Punch  metalcore · thrash metal · grunge
2                   Angel                Aerosmith             rock · hard rock · folk                I Stand Alone                 Godsmack  metalcore · thrash metal · grunge
3                Iron Man            Black Sabbath             rock · hard rock · folk              Within My Blood            Skeletonwitch  metalcore · thrash metal · grunge
4             Crazy Train            Ozzy Osbourne             rock · hard rock · folk          Tears Of The Dragon          Bruce Dickinson  metalcore · thrash metal · grunge
5           Ace of spades                Motorhead    heavy metal · nu metal · hip-hop        Death Before Dishonor  Five Finger Death Punch  metalcore · thrash metal · grunge
6     Rainbow In The Dark                      Dio             rock · hard rock · folk                  Motorbreath                Metallica  metalcore · thrash metal · grunge
7             Hard to See  Five Finger Death Punch   metalcore · thrash metal · grunge  (Anaesthesia) Pulling Teath                Metallica  metalcore · thrash metal · grunge
8           I Stand Alone                 Godsmack   metalcore · thrash metal · grunge                     Thrasher                    Evile  metalcore · thrash metal · grunge
9         Within My Blood            Skeletonwitch   metalcore · thrash metal · grunge           Mercy Is No Virtue                  HEATHEN  metalcore · thrash metal · grunge
10            Only Lonely                 Bon Jovi             rock · hard rock · folk                     New Song          Nuclear Assault  metalcore · thrash metal · grunge

## 6. Evaluación offline

El grafo no trae un conjunto de prueba de gustos de usuarios. Se usa entonces la técnica estándar de
**ocultar relaciones conocidas** (*link hiding*): se borran de la proyección, se recalculan las comunidades sin ellas
para que no haya fuga de información, y se mide si el método vuelve a encontrar los nodos ocultos.

### 6.1 Canción → canciones similares (`SIMILAR_TO` ocultas)

**Diseño del experimento:**
- **Semillas:** 200 canciones con al menos 6 similares.
- **Lo que se oculta:** el 30 % de sus relaciones `SIMILAR_TO`, en ambos sentidos.
- **Métrica:** *recall@k*, la fracción de canciones ocultas que aparece entre los `k` primeros candidatos. Se excluyen
  la semilla y sus similares visibles.

**Líneas base:**
- canciones del mismo artista;
- popularidad, medida por grado;
- solo Louvain: canciones de la misma comunidad, ordenadas por grado.

Como referencia, el azar logra recall@50 ≈ 0,05 %.

In [20]:
def contexto_sin(excluir, pesos=PESOS, comunidades=True):
    ctx = {'G': construir_grafo(DATOS, pesos, excluir)}
    if comunidades:
        ctx['A'] = construir_cografo(DATOS, excluir)
        ctx['com'], ctx['comms'] = detectar_comunidades(ctx['A'])
        ctx['fuerza'] = dict(ctx['A'].degree(weight='weight'))
    return ctx

rng = random.Random(SEMILLA)
SIMILARES = defaultdict(list)
for f in DATOS['SIMILAR_TO']:
    SIMILARES[f['s']].append(f['o'])
SEMILLAS_EVAL = sorted(t for t, v in SIMILARES.items() if len(v) >= 6)
rng.shuffle(SEMILLAS_EVAL)
SEMILLAS_EVAL = SEMILLAS_EVAL[:200]
OCULTAS = {}
for t in SEMILLAS_EVAL:
    v = sorted(SIMILARES[t]); rng.shuffle(v)
    OCULTAS[t] = set(v[:max(2, len(v) * 3 // 10)])
EXCLUIR_SIM = {p for t, hs in OCULTAS.items() for h in hs for p in ((t, h), (h, t))}
print(f'{len(SEMILLAS_EVAL)} semillas, {sum(map(len, OCULTAS.values()))} relaciones ocultas '
      f'({sum(ARTISTA_DE[h] == ARTISTA_DE[t] for t, hs in OCULTAS.items() for h in hs)} del mismo artista)')

t0 = time.time()
ctx_ev = contexto_sin(EXCLUIR_SIM)
grado_ev = sorted((n for n in ctx_ev['G'] if TIPO.get(n) == 'Track'), key=lambda n: -ctx_ev['G'].degree(n, weight='weight'))
canciones_artista = defaultdict(list)
for t in grado_ev:
    canciones_artista[ARTISTA_DE[t]].append(t)

K = (10, 50)
res = defaultdict(list)
for t in SEMILLAS_EVAL:
    ocultas, conocidas = OCULTAS[t], {t} | (set(SIMILARES[t]) - OCULTAS[t])
    recall = lambda lista, k: len(ocultas & set(lista[:k])) / len(ocultas)
    filtra = lambda it: [n for n in it if n not in conocidas]
    c0 = ctx_ev['com'].get(ARTISTA_DE[t])
    esc = escena(ctx_ev, {t: 1.0})
    sc = ppr(ctx_ev, {t: 1.0})
    sc_amp = ppr(ctx_ev, ampliar_semillas(ctx_ev, {t: 1.0}, esc))
    listas = {
        'Mismo artista (por grado)': filtra(canciones_artista[ARTISTA_DE[t]])[:50],
        'Popularidad (grado)': filtra(grado_ev[:60]),
        'Solo Louvain (comunidad, por grado)': filtra(n for n in grado_ev if c0 is not None and ctx_ev['com'].get(ARTISTA_DE[n]) == c0)[:50] if c0 is not None else [],
        'PPR': [n for n, _ in rankear_canciones(ctx_ev, sc, conocidas)[:50]],
        f'PPR, máx. {MAX_POR_ARTISTA} por artista': [n for n, _ in rankear_canciones(ctx_ev, sc, conocidas, max_por_artista=MAX_POR_ARTISTA)[:50]],
        'Louvain → PPR (modo escena)': [n for n, _ in rankear_canciones(ctx_ev, sc, conocidas, esc, 'escena')[:50]],
        f'Louvain → PPR (ampliación γ={GAMMA})': [n for n, _ in rankear_canciones(ctx_ev, sc_amp, conocidas)[:50]],
    }
    for nombre, lista in listas.items():
        for k in K:
            res[(nombre, f'recall@{k}')].append(recall(lista, k))
EVAL_SIM = pd.Series({k: sum(v) / len(v) for k, v in res.items()}).unstack().round(3)
print(f'Evaluación en {time.time() - t0:.0f}s')
display(EVAL_SIM)

200 semillas, 770 relaciones ocultas (12 del mismo artista)


Evaluación en 122s


,recall@10,recall@50
Louvain → PPR (ampliación γ=0.3),0.036,0.144
Louvain → PPR (modo escena),0.034,0.156
Mismo artista (por grado),0.010,0.018
PPR,0.032,0.168
"PPR, máx. 2 por artista",0.039,0.213
Popularidad (grado),0.001,0.003
"Solo Louvain (comunidad, por grado)",0.029,0.099


### 6.2 Sensibilidad al peso de `SIMILAR_TO` y a α

Este experimento justifica `PESOS['SIMILAR_TO'] = 10`. Con peso 1, la caminata se reparte por igual entre el artista,
el país y las canciones similares. Al subir el peso, se privilegia la señal más específica. Se usan 100 de las semillas
anteriores.

In [21]:
filas = []
for mult, alpha in [(1, 0.85), (3, 0.85), (10, 0.85), (30, 0.85), (10, 0.6)]:
    ctx_p = contexto_sin(EXCLUIR_SIM, {**PESOS, 'SIMILAR_TO': mult}, comunidades=False)
    if True:
        r10, r50 = [], []
        for t in SEMILLAS_EVAL[:100]:
            conocidas = {t} | (set(SIMILARES[t]) - OCULTAS[t])
            lista = [n for n, _ in rankear_canciones(ctx_p, ppr(ctx_p, {t: 1.0}, alpha), conocidas)[:50]]
            r10.append(len(OCULTAS[t] & set(lista[:10])) / len(OCULTAS[t]))
            r50.append(len(OCULTAS[t] & set(lista)) / len(OCULTAS[t]))
        filas.append({'peso SIMILAR_TO': mult, 'alpha': alpha, 'recall@10': sum(r10) / len(r10), 'recall@50': sum(r50) / len(r50)})
display(pd.DataFrame(filas).set_index(['peso SIMILAR_TO', 'alpha']).round(3))

,,recall@10,recall@50
peso SIMILAR_TO,alpha,,
1,0.85,0.014,0.123
3,0.85,0.017,0.155
10,0.85,0.032,0.184
30,0.85,0.038,0.209
10,0.60,0.031,0.201


### 6.3 Tag → canciones (`HAS_TAG` ocultas): aporte de Louvain cuando la preferencia es débil

Para cada tag con al menos 5 canciones se oculta la mitad de sus canciones. Las comunidades se recalculan sin esas
relaciones. Luego se mide si PPR, partiendo del tag, recupera las canciones ocultas con y sin ampliación por la escena.

In [22]:
CANCIONES_TAG = defaultdict(set)
for f in DATOS['HAS_TAG']:
    CANCIONES_TAG[f['o']].add(f['s'])
TAGS_EVAL = sorted(g for g, ts in CANCIONES_TAG.items() if len(ts) >= 5)
rng = random.Random(SEMILLA)
OCULTAS_TAG = {}
for g in TAGS_EVAL:
    ts = sorted(CANCIONES_TAG[g]); rng.shuffle(ts)
    OCULTAS_TAG[g] = set(ts[:len(ts) // 2])
ctx_tag = contexto_sin({(t, g) for g, ts in OCULTAS_TAG.items() for t in ts})
print('Tags evaluados:', ', '.join(NOMBRE[g] for g in TAGS_EVAL))

res = defaultdict(list)
for g in TAGS_EVAL:
    conocidas, ocultas = CANCIONES_TAG[g] - OCULTAS_TAG[g], OCULTAS_TAG[g]
    esc = escena(ctx_tag, {g: 1.0})
    for gamma in (0, 0.3, 0.6):
        pers = ampliar_semillas(ctx_tag, {g: 1.0}, esc, gamma) if gamma else {g: 1.0}
        lista = [n for n, _ in rankear_canciones(ctx_tag, ppr(ctx_tag, pers), conocidas)[:200]]
        for k in (50, 200):
            res[(f'γ = {gamma}' + (' (solo PPR)' if gamma == 0 else ''), f'recall@{k}')].append(len(ocultas & set(lista[:k])) / len(ocultas))
display(pd.Series({k: sum(v) / len(v) for k, v in res.items()}).unstack().round(3))

Tags evaluados: Hip-Hop, black metal, blues, classic rock, country, electronic, female vocalists, hardcore, jazz, metal, pop, punk, rock, soul


,recall@200,recall@50
γ = 0 (solo PPR),0.537,0.344
γ = 0.3,0.445,0.344
γ = 0.6,0.367,0.310


### 6.4 Usuario → próxima reseña (preferencias de CritiqueBrainz)

Para cada usuario con reseñas de al menos 3 artistas se oculta una entidad reseñada, ya sea un álbum o un artista. Luego
se mide si alguna de sus canciones aparece entre los 50 primeros candidatos de PPR, comparado con la popularidad.

In [23]:
RESENAS = defaultdict(list)
for f in DATOS['REVIEWED']:
    RESENAS[f['s']].append(f['o'])
def canciones_de(n):
    t = TIPO.get(n)
    if t == 'Track':
        return {n}
    return {v for v in CTX['G'][n] if TIPO.get(v) == 'Track'}
def artista_principal(n):
    a = artistas_de(CTX, n)
    return a[0] if a else None

rng = random.Random(SEMILLA)
OCULTA_USUARIO = {}
for u, xs in sorted(RESENAS.items()):
    if len({artista_principal(x) for x in xs}) >= 3:
        x = rng.choice(sorted(xs))
        if len({artista_principal(y) for y in xs if y != x} - {artista_principal(x)}) >= 2:
            OCULTA_USUARIO[u] = x
ctx_u = contexto_sin({(u, x) for u, x in OCULTA_USUARIO.items()}, comunidades=False)
populares = Counter(t for u, xs in RESENAS.items() for x in xs if OCULTA_USUARIO.get(u) != x for t in canciones_de(x))
populares = [t for t, _ in populares.most_common()]

aciertos = Counter()
for u, x in OCULTA_USUARIO.items():
    objetivo = canciones_de(x)
    conocidas = set().union(*(canciones_de(y) for y in RESENAS[u] if y != x))
    lista = [n for n, _ in rankear_canciones(ctx_u, ppr(ctx_u, {u: 1.0}), conocidas)[:50]]
    aciertos['PPR'] += bool(objetivo & set(lista))
    aciertos['Popularidad'] += bool(objetivo & set([t for t in populares if t not in conocidas][:50]))
display(pd.DataFrame({'hit@50': {k: round(v / len(OCULTA_USUARIO), 3) for k, v in aciertos.items()},
                      'usuarios': len(OCULTA_USUARIO)}))

,hit@50,usuarios
PPR,0.036,139
Popularidad,0.014,139


**Lectura de la evaluación:**
- **PPR es el mejor generador de candidatos** para una canción. Recupera muchas más similares ocultas que el mismo
  artista o la popularidad, y cientos de veces más que el azar.
- **Diversidad:** limitar a 2 canciones por artista y quitar los duplicados del MSD **mejora** el recall. Esos cupos los
  ocupaban la misma canción repetida u otras canciones del artista semilla, cuando las similares casi nunca son del
  mismo artista. Por eso es la configuración por defecto.
- **Peso de `SIMILAR_TO`:** darle más peso mejora el recall. Se eligió 10 en vez de 30 para no anular las demás señales,
  como artista, álbum y críticos, en las semillas que no tienen similares.
- **Louvain solo** recupera unas 30 veces más que la popularidad. La escena captura afinidad aunque no haya un camino
  directo entre las canciones.
- **Louvain dentro de PPR:**
  - El modo «misma escena» pierde poco recall frente a PPR global y garantiza coherencia de estilo. Queda como opción
    explícita.
  - Ampliar las semillas con la escena **no mejoró** ni con canciones (6.1) ni con tags (6.3): el grafo ya lleva a PPR
    a esos artistas. Por eso está desactivada por defecto. Es un resultado negativo que se reporta tal cual.
- **Usuario → próxima reseña:** es la tarea más difícil. Casi todos los usuarios son críticos de la BBC que reseñan
  discos de todos los géneros, y casi ningún disco tiene más de una reseña. PPR aun así supera a la popularidad, pero el
  valor absoluto es bajo: es una limitación de los datos de preferencias disponibles, no del método.

## 7. Persistencia: recomendaciones por usuario en Neo4j

Para cada usuario se guardan sus 10 canciones candidatas como `(:User)-[:RECOMMENDED {rank, score, community}]->(:Track)`.
Así quedan disponibles desde Cypher y para la generación de explicaciones con RAG del siguiente punto.

In [24]:
t0 = time.time()
filas_rec = []
for u in sorted(usuarios):
    semillas = {u: 1.0}
    esc = escena(CTX, semillas)
    lista = rankear_canciones(CTX, ppr(CTX, semillas), canciones_conocidas(CTX, semillas), esc,
                              max_por_artista=MAX_POR_ARTISTA)[:10]
    tope = lista[0][1] if lista else 1
    filas_rec += [{'u': u, 't': n, 'rank': i, 'score': round(s / tope, 4), 'c': COMUNIDAD.get(ARTISTA_DE[n])}
                  for i, (n, s) in enumerate(lista, 1)]
print(f'{len(filas_rec):,} recomendaciones para {len(usuarios)} usuarios en {time.time() - t0:.0f}s')

run_cypher('MATCH (:User)-[r:RECOMMENDED]->(:Track) DELETE r')
cargar_en_lotes('Relaciones :RECOMMENDED', '''UNWIND $rows AS row
    MATCH (u:User {uri: row.u}) MATCH (t:Track {uri: row.t})
    MERGE (u)-[r:RECOMMENDED]->(t)
    SET r.rank = row.rank, r.score = row.score, r.community = row.c, r.method = 'ppr+louvain' ''', filas_rec)

3,870 recomendaciones para 387 usuarios en 123s


Conexión perdida (ServiceUnavailable); reconectando 1/14 en 5s...


✓ Relaciones :RECOMMENDED                     3,870  (3.5s)


In [25]:
n_rec = run_cypher('MATCH (:User)-[r:RECOMMENDED]->(:Track) RETURN count(r) AS n')[0]['n']
n_com = run_cypher('MATCH (a:Artist) WHERE a.community IS NOT NULL RETURN count(a) AS n')[0]['n']
assert n_rec == len(filas_rec), (n_rec, len(filas_rec))
assert n_com == len(COMUNIDAD), (n_com, len(COMUNIDAD))
print(f'✓ RECOMMENDED en Neo4j: {n_rec:,} | ✓ artistas con comunidad: {n_com:,}')

# Consulta de ejemplo desde Cypher: recomendaciones de un usuario, con la escena y lo que reseñó de esa escena
display(pd.DataFrame(run_cypher('''
    MATCH (u:User {uri: $u})-[r:RECOMMENDED]->(t:Track)-[:SUNG_BY]->(a:Artist)
    OPTIONAL MATCH (u)<-[:WRITTEN_BY]-(:Review)-[:REVIEWS_ALBUM]->(:Album)-[:RELEASED_BY]->(b:Artist {community: a.community})
    RETURN r.rank AS rank, t.title AS cancion, a.name AS artista, a.community AS comunidad,
           collect(DISTINCT b.name)[..3] AS `reseñó de la misma escena`
    ORDER BY rank''', u=usuario_demo)))
totales = run_cypher('CALL () { MATCH (n) RETURN count(n) AS nodos } CALL () { MATCH ()-[r]->() RETURN count(r) AS rels } RETURN nodos, rels')[0]
print(f'Totales del grafo: {totales["nodos"]:,} nodos y {totales["rels"]:,} relaciones (límite AuraDB Free: 200k / 400k)')

✓ RECOMMENDED en Neo4j: 3,870 | ✓ artistas con comunidad: 4,747


,rank,cancion,artista,comunidad,reseñó de la misma escena
0,1,Rock Steady,Bad Company,11,[]
1,2,Kryptonite,3 Doors Down,0,"[AFI, Aerosmith, Adema]"
2,3,Here Without You,3 Doors Down,0,"[AFI, Aerosmith, Adema]"
3,4,Time Is On My Side,The Rolling Stones,11,[]
4,5,3 Leaf Loser,12 Stones,0,"[AFI, Aerosmith, Adema]"
5,6,In Closing,12 Stones,0,"[AFI, Aerosmith, Adema]"
6,7,Can't Get Enough,Bad Company,11,[]
7,8,Angel,Aerosmith,0,"[AFI, Aerosmith, Adema]"
8,9,Way Beyond,32 Leaves,0,"[AFI, Aerosmith, Adema]"
9,10,Reconsider Everything,311,0,"[AFI, Aerosmith, Adema]"


Totales del grafo: 137,446 nodos y 151,553 relaciones (límite AuraDB Free: 200k / 400k)


## 8. Resumen

| Algoritmo | Parte del grafo | Información que produce | Contribución a los candidatos |
|---|---|---|---|
| **WCC** (diagnóstico) | Proyección completa, con y sin país o usuarios | Componentes conexas | Muestra que sin el país la mayoría de canciones queda aislada. Justifica incluir el país con peso bajo y trabajar Louvain a nivel de artista |
| **Louvain** | Co-grafo de artistas: similitud entre canciones, tags compartidos, críticos en común, género/instrumento/persona del bonus | Comunidad (escena) por artista y modularidad. Se guarda en `Artist.community` | Escena de la preferencia. Modo «misma escena» (Louvain selecciona, PPR ordena). Anotación y explicación de cada candidato. Por sí solo recupera ~30 veces más que la popularidad. La ampliación de semillas se evaluó y no mejoró, así que es opcional |
| **Personalized PageRank** | Grafo heterogéneo completo y ponderado (canciones, artistas, álbumes, tags, usuarios, países, bonus) | Puntuación de cercanía de cada nodo a las semillas | Ranking de canciones, artistas y álbumes candidatos, con un camino explicativo. Es el mejor generador en la evaluación por ocultamiento, sobre todo con diversidad (2 por artista) |

**Salidas para los siguientes puntos:**
- `recomendar(...)` devuelve candidatos con score, escena y camino explicativo.
- En Neo4j quedan `Artist.community` y `(:User)-[:RECOMMENDED]->(:Track)`.
- Cada candidato puede enriquecerse con los fragmentos de reseñas del índice vectorial del punto 2
  (`recuperar_para_canciones`) para generar explicaciones con RAG.

In [26]:
driver.close()